In [2]:
from datetime import datetime

import arxiv
from loguru import logger
from pyspark.sql import SparkSession
from pyspark.sql.types import ArrayType, LongType, StringType, StructField, StructType

from arxiv_curator.config import get_env, load_config

In [3]:
spark = SparkSession.builder.getOrCreate()

# Load config
env = get_env(spark)
cfg = load_config("../project_config.yml", env)

CATALOG = cfg.catalog
SCHEMA = cfg.schema
TABLE_NAME = "arxiv_papers"

# Create schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
logger.info(f"Schema {CATALOG}.{SCHEMA} ready")

2026-03-18 10:55:13.786 | INFO     | __main__:<module>:13 - Schema site.ai_library_dev ready


In [4]:
def fetch_arxiv_papers(query: str = "cat:cs.AI OR cat:cs.LG", max_results: int = 100):
    """
    Fetch arXiv papers using the arXiv API.

    Args:
        query: arXiv search query (default: AI and ML papers)
        max_results: Maximum number of papers to fetch

    Returns:
        List of paper metadata dictionaries
    """
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending
    )

    papers = []
    for result in client.results(search):
        paper = {
            "arxiv_id": result.entry_id.split("/")[-1],
            "title": result.title,
            "authors": [author.name for author in result.authors],  # Array to match reference code
            "summary": result.summary,
            "published": int(result.published.strftime("%Y%m%d%H%M")),  # Long to match reference code
            "updated": result.updated.isoformat() if result.updated else None,
            "categories": ", ".join(result.categories),
            "pdf_url": result.pdf_url,
            "primary_category": result.primary_category,
            "ingestion_timestamp": datetime.now().isoformat(),
            "processed": None,  # Will be set in Lecture 2.2
            "volume_path": None  # Will be set in Lecture 2.2
        }
        papers.append(paper)

    return papers

logger.info("Fetching arXiv papers...")
papers = fetch_arxiv_papers(query="cat:cs.AI OR cat:cs.LG", max_results=50)
logger.info(f"Fetched {len(papers)} papers")
logger.info("Sample paper:")
logger.info(f"Title: {papers[0]['title']}")
logger.info(f"Authors: {papers[0]['authors']}")
logger.info(f"arXiv ID: {papers[0]['arxiv_id']}")
logger.info(f"PDF URL: {papers[0]['pdf_url']}")

2026-03-18 10:55:20.434 | INFO     | __main__:<module>:40 - Fetching arXiv papers...
2026-03-18 10:55:21.393 | INFO     | __main__:<module>:42 - Fetched 50 papers
2026-03-18 10:55:21.393 | INFO     | __main__:<module>:43 - Sample paper:
2026-03-18 10:55:21.393 | INFO     | __main__:<module>:44 - Title: Demystifing Video Reasoning
2026-03-18 10:55:21.393 | INFO     | __main__:<module>:45 - Authors: ['Ruisi Wang', 'Zhongang Cai', 'Fanyi Pu', 'Junxiang Xu', 'Wanqi Yin', 'Maijunxian Wang', 'Ran Ji', 'Chenyang Gu', 'Bo Li', 'Ziqi Huang', 'Hokin Deng', 'Dahua Lin', 'Ziwei Liu', 'Lei Yang']
2026-03-18 10:55:21.393 | INFO     | __main__:<module>:46 - arXiv ID: 2603.16870v1
2026-03-18 10:55:21.393 | INFO     | __main__:<module>:47 - PDF URL: https://arxiv.org/pdf/2603.16870v1


In [5]:
schema = StructType([
    StructField("arxiv_id", StringType(), False),
    StructField("title", StringType(), False),
    StructField("authors", ArrayType(StringType()), True),  # Array to match reference code
    StructField("summary", StringType(), True),
    StructField("published", LongType(), True),  # Long to match reference code
    StructField("updated", StringType(), True),
    StructField("categories", StringType(), True),
    StructField("pdf_url", StringType(), True),
    StructField("primary_category", StringType(), True),
    StructField("ingestion_timestamp", StringType(), True),
    StructField("processed", LongType(), True),  # Long to match reference code
    StructField("volume_path", StringType(), True)  # Will be set in Lecture 2.2
])

# Create DataFrame
df = spark.createDataFrame(papers, schema=schema)

# Write to Delta table
table_path = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(table_path)

logger.info(f"Created Delta table: {table_path}")
logger.info(f"Records: {df.count()}")

2026-03-18 10:55:37.376 | INFO     | __main__:<module>:28 - Created Delta table: site.ai_library_dev.arxiv_papers
2026-03-18 10:55:38.354 | INFO     | __main__:<module>:29 - Records: 50


In [6]:
# Verify the Data

# Read back the table
papers_df = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}")

logger.info(f"Table: {CATALOG}.{SCHEMA}.{TABLE_NAME}")
logger.info(f"Total papers: {papers_df.count()}")
logger.info("Schema:")
papers_df.printSchema()

logger.info("Sample records:")
papers_df.select("arxiv_id", "title", "primary_category", "published").show(5, truncate=50)

2026-03-18 10:56:09.942 | INFO     | __main__:<module>:6 - Table: site.ai_library_dev.arxiv_papers
2026-03-18 10:56:10.734 | INFO     | __main__:<module>:7 - Total papers: 50
2026-03-18 10:56:10.741 | INFO     | __main__:<module>:8 - Schema:
2026-03-18 10:56:10.985 | INFO     | __main__:<module>:11 - Sample records:


root
 |-- arxiv_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- authors: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- summary: string (nullable = true)
 |-- published: long (nullable = true)
 |-- updated: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- pdf_url: string (nullable = true)
 |-- primary_category: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- processed: long (nullable = true)
 |-- volume_path: string (nullable = true)

+------------+--------------------------------------------------+----------------+------------+
|    arxiv_id|                                             title|primary_category|   published|
+------------+--------------------------------------------------+----------------+------------+
|2603.16870v1|                       Demystifing Video Reasoning|           cs.CV|202603171759|
|2603.16867v1|                   Efficient Reasoning on the Edge|   

In [7]:
logger.info("Papers by primary category:")
papers_df.groupBy("primary_category").count().orderBy("count", ascending=False).show()

logger.info("Most recent papers:")
papers_df.select("title", "published", "arxiv_id") \
    .orderBy("published", ascending=False) \
    .show(5, truncate=60)

2026-03-18 10:56:21.410 | INFO     | __main__:<module>:1 - Papers by primary category:
2026-03-18 10:56:22.056 | INFO     | __main__:<module>:4 - Most recent papers:


+----------------+-----+
|primary_category|count|
+----------------+-----+
|           cs.LG|   19|
|           cs.AI|   12|
|           cs.CV|    7|
|           cs.RO|    4|
|           cs.CL|    2|
|         math.NA|    1|
|         stat.ML|    1|
|           cs.SE|    1|
|           cs.GT|    1|
|           cs.DC|    1|
|         math.DS|    1|
+----------------+-----+

+------------------------------------------------------------+------------+------------+
|                                                       title|   published|    arxiv_id|
+------------------------------------------------------------+------------+------------+
|ManiTwin: Scaling Data-Generation-Ready Digital Object Da...|202603171759|2603.16866v1|
|SparkVSR: Interactive Video Super-Resolution via Sparse K...|202603171759|2603.16864v1|
|                                 Demystifing Video Reasoning|202603171759|2603.16870v1|
|MessyKitchens: Contact-rich object-level 3D scene reconst...|202603171759|2603.16868v1|
|